## Dataset Preparation

#### imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import utils

In [7]:
DISASTER_DATA_DIR = Path("../datasets/storm_events")
COND_FEATS_PATH = Path("../datasets/disasters_new_feats/all_new_feats.csv")

### Loading

In [8]:
disaster_df = utils.load_disaster_dataset(DISASTER_DATA_DIR)
feats_df = pd.read_csv(COND_FEATS_PATH)

In [9]:
print(disaster_df.columns)
print(feats_df.columns)

Index(['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH',
       'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS',
       'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME',
       'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME',
       'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT',
       'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE',
       'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE',
       'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE',
       'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE',
       'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH',
       'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
       'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE'],
      dtype='str')
Index(['PRECIPITATION', 'TMIN', 'TMAX', 'BEGIN_LAT', 'BEGIN_LON', 'ELEVATION',
       'SLOPE', 'COV_BARREN', 'COV_CULTIVATED', 'COV_VEG

### Disaster processing

In [10]:
# reduce the disaster dataset to only the relevant columns
cols_to_keep = ["EVENT_TYPE", "BEGIN_DATE_TIME", "END_DATE_TIME", "INJURIES_DIRECT", "INJURIES_INDIRECT", "DEATHS_DIRECT",
                     "DEATHS_INDIRECT", "DAMAGE_PROPERTY", "DAMAGE_CROPS", "MAGNITUDE", "BEGIN_LAT", "BEGIN_LON", "END_LAT", "END_LON"] # "STATE", "FLOOD_CAUSE", "TOR_F_SCALE", "TOR_LENGTH", "TOR_WIDTH", "END_AZIMUTH", "END_LOCATION", 
disaster_df = disaster_df[cols_to_keep]

In [11]:
# remove unpredictable disaster types
unpredictable_events = ['High Surf', 'Sneakerwave', 'Seiche', 'Dense Smoke', 'Rip Current', 'Tropical Depression', 'Tsunami',
                    'Marine Tropical Depression', 'Volcanic Ashfall', 'Astronomical Low Tide', 'Dust Devil', 'Storm Surge/Tide']

disaster_df = disaster_df[~disaster_df['EVENT_TYPE'].isin(unpredictable_events)]

In [12]:
# aggregate events into broader categories
disaster_df['EVENT_GROUP'] = disaster_df['EVENT_TYPE'].replace(utils.disaster_events_group_map)
disaster_df.drop(columns=['EVENT_TYPE'], inplace=True)

In [13]:
# aggregate y cols into damages and casualties
disaster_df['DAMAGES'] = disaster_df['DAMAGE_PROPERTY'].apply(utils.damage_to_numeric) + disaster_df['DAMAGE_CROPS'].apply(utils.damage_to_numeric)
disaster_df['CASUALTIES'] = disaster_df['INJURIES_DIRECT'].fillna(0) + disaster_df['INJURIES_INDIRECT'].fillna(0) + disaster_df['DEATHS_DIRECT'].fillna(0) + disaster_df['DEATHS_INDIRECT'].fillna(0)

disaster_df.drop(columns=['DAMAGE_PROPERTY', 'DAMAGE_CROPS'], inplace=True)
disaster_df.drop(columns=['INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT'], inplace=True)

In [14]:
# change END_DATE_TIME to duration in hours
disaster_df['END_DATE_TIME'] = pd.to_datetime(disaster_df['END_DATE_TIME'])
disaster_df['BEGIN_DATE_TIME'] = pd.to_datetime(disaster_df['BEGIN_DATE_TIME'])
disaster_df['DURATION_HOURS'] = (disaster_df['END_DATE_TIME'] - disaster_df['BEGIN_DATE_TIME']).dt.total_seconds() / 3600
disaster_df.drop(columns=['END_DATE_TIME'], inplace=True)

C:\Users\matti\AppData\Local\Temp\ipykernel_11964\2042865307.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disaster_df['END_DATE_TIME'] = pd.to_datetime(disaster_df['END_DATE_TIME'])
C:\Users\matti\AppData\Local\Temp\ipykernel_11964\2042865307.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disaster_df['BEGIN_DATE_TIME'] = pd.to_datetime(disaster_df['BEGIN_DATE_TIME'])


In [15]:
# split MAGNITUDE into HAIL_SIZE and WIND_SPEED
disaster_df['HAIL_SIZE'] = disaster_df['MAGNITUDE'].where(disaster_df['EVENT_GROUP'] == 'Hail', 0)
disaster_df['WIND_SPEED'] = disaster_df['MAGNITUDE'].where(disaster_df['EVENT_GROUP'] != 'Hail', 0)
disaster_df['WIND_SPEED'] = disaster_df['WIND_SPEED'].fillna(0)
disaster_df.drop(columns=['MAGNITUDE'], inplace=True)

In [16]:
# encode datetime
disaster_df['TIME_DAY_SIN'] = np.sin(2 * np.pi * disaster_df['BEGIN_DATE_TIME'].dt.dayofyear / 365.25)
disaster_df['TIME_DAY_COS'] = np.cos(2 * np.pi * disaster_df['BEGIN_DATE_TIME'].dt.dayofyear / 365.25)
disaster_df['TIME_YEAR_NORM'] = (disaster_df['BEGIN_DATE_TIME'].dt.year - disaster_df['BEGIN_DATE_TIME'].dt.year.min()) / (disaster_df['BEGIN_DATE_TIME'].dt.year.max() - disaster_df['BEGIN_DATE_TIME'].dt.year.min())
disaster_df.drop(columns=['BEGIN_DATE_TIME'], inplace=True)

In [17]:
# remove coords cols (already have them in the feats dataset)
disaster_df = disaster_df.drop(columns=["BEGIN_LAT", "BEGIN_LON", "END_LAT", "END_LON"])

In [18]:
disaster_df.head()

,EVENT_GROUP,DAMAGES,CASUALTIES,DURATION_HOURS,HAIL_SIZE,WIND_SPEED,TIME_DAY_SIN,TIME_DAY_COS,TIME_YEAR_NORM
0,Heavy Rain,2000.0,0,0.0,0.0,0.0,-0.244772,-0.969581,0.0
1,Wind,0.0,0,0.0,0.0,50.0,-0.244772,-0.969581,0.0
2,Wind,0.0,0,0.0,0.0,50.0,-0.261414,-0.965227,0.0
3,Wind,0.0,0,0.0,0.0,50.0,-0.126528,-0.991963,0.0
4,Wind,0.0,0,0.0,0.0,60.0,-0.126528,-0.991963,0.0


### Merging

In [19]:
df = pd.merge(disaster_df, feats_df, left_index=True, right_index=True)
df.head()

,EVENT_GROUP,DAMAGES,CASUALTIES,DURATION_HOURS,HAIL_SIZE,WIND_SPEED,TIME_DAY_SIN,TIME_DAY_COS,TIME_YEAR_NORM,PRECIPITATION,...,SLOPE,COV_BARREN,COV_CULTIVATED,COV_VEGETATION,COV_FOREST,COV_WATER,COV_SNOW_ICE,COV_URBAN,RIVER_DISTANCE,SEA_DISTANCE
0,Heavy Rain,2000.0,0,0.0,0.0,0.0,-0.244772,-0.969581,0.0,1133.73,...,5.62,0.0,0.157,0.039,0.708,0.012,0.0,0.085,1248.1,237.1
1,Wind,0.0,0,0.0,0.0,50.0,-0.244772,-0.969581,0.0,1144.69,...,6.52,0.0,0.173,0.048,0.702,0.007,0.0,0.070,528.2,236.5
2,Wind,0.0,0,0.0,0.0,50.0,-0.261414,-0.965227,0.0,1433.45,...,39.69,0.0,0.056,0.006,0.936,0.001,0.0,0.001,431.3,363.6
3,Wind,0.0,0,0.0,0.0,50.0,-0.126528,-0.991963,0.0,1442.06,...,11.47,0.0,0.379,0.023,0.527,0.069,0.0,0.001,842.2,462.1
4,Wind,0.0,0,0.0,0.0,60.0,-0.126528,-0.991963,0.0,1327.57,...,15.72,0.0,0.405,0.016,0.567,0.006,0.0,0.005,1563.3,457.8


### Saving

In [20]:
df.to_csv("../datasets/disasters_merged_all_feats.csv", index=False)